# Module 12 lab · Inference and evals: measuring models honestly

The code from every lesson of [this module](https://aicourses.dev/research/inference-and-evals), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 12.2 Fast generation: the KV cache, batching and vLLM

When a model writes one token at a time, it redoes the same work for every earlier token. The KV cache saves that work, grouped-query attention makes the cache smaller, and batching serves many requests together. vLLM is a tool that does all of this, for the thousands of answers later lessons need.

[Read the lesson](https://aicourses.dev/research/inference-and-evals/kv-cache-and-vllm)

### The cache in code

The block below runs one attention head on 100 made-up token vectors, both ways. In a real model each new vector comes from the token just sampled; here they are random, which is enough to compare the two loops.

In [ ]:
import torch

torch.manual_seed(0)
d = 8                                  # size of each token's vector
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))
tokens = torch.randn(100, d)           # 100 made-up token vectors

def attend(q, K, V):
    w = torch.softmax(q @ K.T / d**0.5, dim=-1)
    return w @ V
# No cache: rebuild keys and values for every earlier token, each step
slow, work = [], 0
for t in range(len(tokens)):
    K, V = tokens[:t + 1] @ Wk, tokens[:t + 1] @ Wv
    work += t + 1
    slow.append(attend(tokens[t] @ Wq, K, V))
# Cache: make keys and values for the new token only, keep the rest
Ks, Vs, fast = [], [], []
for t in range(len(tokens)):
    Ks.append(tokens[t] @ Wk)
    Vs.append(tokens[t] @ Wv)
    K, V = torch.stack(Ks), torch.stack(Vs)
    fast.append(attend(tokens[t] @ Wq, K, V))
print(work, len(Ks))                                   # 5050 100
print(torch.allclose(torch.stack(slow), torch.stack(fast)))  # True

### How big the notes get

Take Qwen2.5-0.5B, a small open model with 0.49 billion parameters. Its published config lists 24 layers, 14 query heads, 2 key-value heads and a hidden size (the width of each token’s vector) of 896, so each head has 896 ÷ 14 = 64 numbers. Then 2 × 24 × 2 × 64 × 2 = 12,288 bytes, or 12 KiB, per token. A KiB is 1,024 bytes, a MiB 1,024 KiB and a GiB 1,024 MiB, so the code divides by $2^{20}$ to get MiB.

In [ ]:
layers, kv_heads, head_dim = 24, 2, 64  # from Qwen2.5-0.5B's config
bytes_each = 2                          # one BF16 number is 2 bytes

per_token = 2 * layers * kv_heads * head_dim * bytes_each
print(per_token)                        # 12288
print(per_token * 32_768 / 2**20)       # 384.0 MiB for 32,768 tokens

full = 2 * layers * 14 * head_dim * bytes_each   # one K, V per head
print(full / per_token)                 # 7.0

### Serving many people at once

vLLM needs an NVIDIA GPU, so in Colab choose Runtime > Change runtime type and pick a GPU. Install it in a cell with `!pip install vllm`, where the `!` runs the line as a command instead of Python, then run:

The lesson installs this first:

In [ ]:
!pip install vllm

This cell needs a GPU runtime.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model="Qwen/Qwen2.5-0.5B",       # a base model
          dtype="half")                    # FP16, for the T4
params = SamplingParams(n=8, temperature=0.8, top_p=0.95,
                        max_tokens=128, seed=0)
prompts = [f"Question: What is {a} + {2 * a}?\nAnswer:"
           for a in range(1, 101)]
outputs = llm.generate(prompts, params)    # one call, all prompts
for out in outputs[:2]:
    print(out.prompt)
    print(out.outputs[0].text)             # first of its 8 answers

## 12.3 Benchmarks: what they measure, and reproducing a published number

The same model can score differently depending on how you ask and how you read its answers, or because the questions leaked into its training data. You reproduce a published score and explain the gap, a common eval task in this job.

[Read the lesson](https://aicourses.dev/research/inference-and-evals/benchmarks)

### Same model, different numbers

The answer reader alone can do this. Here are four answers a model might write to GSM8K-style questions, read by two rules much like the harness’s “strict” and “flexible” readers:

In [ ]:
import re

answers = [  # (what the model wrote, the right answer)
    ("3 + 4 = 7 apples.\n#### 7", "7"),
    ("3 + 4 = 7 apples. The answer is 7.", "7"),
    ("5 boxes of 12 hold 60 eggs.\n#### 60", "60"),
    ("The answer is 60 eggs, from 5 boxes.", "60"),
]

def strict(text):            # only a number written after ####
    found = re.findall(r"#### (-?[\d,]+)", text)
    return found[0] if found else None

def flexible(text):          # the last number anywhere
    found = re.findall(r"-?\d[\d,]*", text)
    return found[-1] if found else None

for reader in (strict, flexible):
    right = [reader(text) == gold for text, gold in answers]
    print(reader.__name__, right, sum(right) / len(right))
# strict [True, False, True, False] 0.5
# flexible [True, True, True, False] 0.75

### Reproduce a published number

Qwen’s announcement of its Qwen2.5 models lists, for the 0.5-billion-parameter base model, 47.5% on MMLU with 5 shots and 41.6% on GSM8K with 4 shots. Reproduce one with **lm-eval**, EleutherAI’s free evaluation harness. It downloads the model and needs a GPU, so in Colab choose Runtime > Change runtime type and pick a GPU, then run this cell. The `!` runs each line as a command rather than as Python:

This cell needs a GPU runtime.

In [ ]:
!pip install "lm_eval[hf]"
!lm_eval --model hf \
  --model_args pretrained=Qwen/Qwen2.5-0.5B,dtype=float16 \
  --tasks gsm8k --num_fewshot 4 --batch_size 16 \
  --output_path results/ --log_samples

## 12.4 Error bars: how much a score could move by chance

A score on 500 questions would change on a different 500. The standard error, built from the spread you met in Module 3, says by how much, and error bars draw it. You guess whether 72.1 against 71.4 is a real gap before you check.

[Read the lesson](https://aicourses.dev/research/inference-and-evals/error-bars)

### The standard error, by formula

Check the formula by simulation. This draws 10,000 tests of 500 questions from a model whose true skill is 72%:

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
p, n = 0.72, 500                  # true skill, questions per test
right = rng.random((10_000, n)) < p   # 10,000 tests of 500 questions
scores = right.mean(axis=1)
print(scores[:4])                 # [0.69  0.718 0.718 0.748]
print(scores.std())               # 0.02047312675289244
print(np.sqrt(p * (1 - p) / n))   # 0.020079840636817812

### From standard error to error bar

What does “95%” promise? Not that the true skill is in this one interval with chance 95%. It promises that if you repeated the whole test many times with fresh questions, and built an interval each time this way, about 95% of those intervals would contain the true skill. You can check that promise directly:

In [ ]:
se = np.sqrt(scores * (1 - scores) / n)   # each test's own estimate
low, high = scores - 1.96 * se, scores + 1.96 * se
print(np.mean((low <= p) & (p <= high)))  # 0.942

### Settling 72.1 against 71.4

Now draw both scores with their bars:

In [ ]:
import matplotlib.pyplot as plt

names = ["Model A", "Model B"]
acc = np.array([0.721, 0.714])     # scores on 500 questions each
se = np.sqrt(acc * (1 - acc) / 500)
print(se.round(4))                 # [0.0201 0.0202]
print((acc - 1.96 * se).round(3), (acc + 1.96 * se).round(3))
# [0.682 0.674] [0.76  0.754]
plt.bar(names, acc, yerr=1.96 * se, capsize=8)
plt.ylim(0.6, 0.8)
plt.ylabel("accuracy")
plt.show()

## 12.5 Comparing two models fairly: paired tests, seeds and the bootstrap

Score both models on the same questions and look at the differences: this gives much tighter answers. The bootstrap finds error bars by resampling your own results, with no formula.

[Read the lesson](https://aicourses.dev/research/inference-and-evals/comparing-models-fairly)

### Paired differences in code

The simulation below gives every question an “ease”, shared by both models, and makes model B slightly weaker:

In [ ]:
import numpy as np

rng = np.random.default_rng(8)
n = 500
ease = 2 * rng.normal(size=n)          # how easy each question is
a = ease + rng.normal(size=n) > -1.3   # model A: right or wrong
b = ease + rng.normal(size=n) > -1.0   # model B: right or wrong
a, b = a.astype(float), b.astype(float)
print(a.mean(), b.mean())              # 0.704 0.652
print(np.corrcoef(a, b)[0, 1])         # 0.6024197036341227

d = a - b                              # +1, 0 or -1 per question
se_two = np.sqrt(a.var() / n + b.var() / n)   # as if separate tests
se_pair = d.std() / np.sqrt(n)                # from the differences
print(d.mean(), se_two, se_pair)
# 0.052 0.029505253769456043 0.018616981495398227

### The bootstrap: error bars by resampling

In [ ]:
pick = rng.integers(0, n, size=(10_000, n))  # 10,000 resamples
boot = d[pick].mean(axis=1)            # mean difference of each
print(np.percentile(boot, [2.5, 97.5]))      # [0.016 0.088]

### How many questions do you need?

Check the promise by simulation, with 10,000 paired tests of 543 questions and a true gap of 5 points:

In [ ]:
rng = np.random.default_rng(2)
n, tests = 543, 10_000
# each question: only A right (+1), only B right (-1), or a tie (0)
chance = [0.1129, 0.0629, 0.8242]
d = rng.choice([1, -1, 0], p=chance, size=(tests, n))
gap = d.mean(axis=1)
se = d.std(axis=1) / np.sqrt(n)
print(np.mean(gap - 1.96 * se > 0))   # 0.803

## 12.6 LLM-as-judge: building one and measuring its biases

Some answers, like essays, have no single right answer, so a strong model grades them. Judges have habits, such as preferring the first or the longest answer, and you measure them before trusting the grades. Later these grades become rewards.

[Read the lesson](https://aicourses.dev/research/inference-and-evals/llm-as-judge)

### Building a judge

A **pairwise** judge sees one question and two answers and names the better one. A **single-answer** judge gives one answer a score, say 1 to 10. Pairwise suits comparing two models directly; single scores are cheaper when there are many models, because each answer is graded once. Here is a pairwise prompt and the code that reads the verdict:

In [ ]:
import re

TEMPLATE = """You are judging two answers to the same question.
Question: {question}

[Answer A]
{a}

[Answer B]
{b}

Which answer is more helpful and correct? Explain briefly, then
end with exactly [[A]], [[B]] or [[C]] for a tie."""

def verdict(judge_text):
    found = re.findall(r"\[\[([ABC])\]\]", judge_text)
    return found[-1] if found else None    # None: judge broke format

print(verdict("A explains the physics; B does not. [[A]]"))   # A
print(verdict("Both are fine."))                              # None

To run it, send the filled-in template to an instruction-tuned model, one fine-tuned to follow requests (the “Instruct” in its name). With Hugging Face Transformers, on a Colab GPU:

In [ ]:
from transformers import pipeline

judge = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")
messages = [{"role": "user", "content": TEMPLATE.format(
    question="Why is the sky blue?",
    a="Air scatters short blue wavelengths of sunlight the most.",
    b="Because it reflects the ocean.")}]
reply = judge(messages, max_new_tokens=200)[0]["generated_text"]
print(verdict(reply[-1]["content"]))

### Three habits to measure

The swap test is easy to simulate. Below, a made-up judge sees 400 pairs, each with a hidden true quality difference, plus a pull of 0.6 towards whichever answer comes first and a little noise. It is shown every pair in both orders:

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
pairs = 400
quality = rng.normal(size=pairs)      # > 0 means answer X is better
tilt = 0.6                            # the judge's pull to the first

# Order 1 shows X first; order 2 shows Y first
x_first = quality + tilt + rng.normal(0, 0.5, pairs) > 0
y_first = -quality + tilt + rng.normal(0, 0.5, pairs) > 0
picks_x_1 = x_first                   # in order 1, first means X
picks_x_2 = ~y_first                  # in order 2, second means X

consistent = picks_x_1 == picks_x_2
print("consistent:", consistent.mean())                    # 0.56
print("first both times:", (x_first & y_first).mean())     # 0.4375
print("second both times:", (~x_first & ~y_first).mean())  # 0.0025

`rng.normal(0, 0.5, pairs)` draws 400 bell-shaped numbers with mean 0 and standard deviation 0.5, the noise, and `~` flips each True to False and back. Only 56% of verdicts survive the swap, and almost all the rest went to whichever answer was first. These three numbers are exactly what you report for your own judge. Now apply the fix the MT-Bench study used: count a win only when the judge picks the same answer in both orders, and call everything else a tie.

In [ ]:
truth = quality > 0                   # is X really the better one?
print("one order, right:", (picks_x_1 == truth).mean())   # 0.7475
decided = consistent                  # inconsistent pairs become ties
print("swapped, right when decided:",
      (picks_x_1[decided] == truth[decided]).mean())      # 0.96875

## 12.7 Writing your own eval with Inspect

An eval is a set of questions, a way to ask a model, and a way to score its answers. Inspect is a free tool for writing them, used by labs and government AI institutes. Module 13’s fine-tune is judged with it.

[Read the lesson](https://aicourses.dev/research/inference-and-evals/inspect-evals)

### Your first task

This task asks three addition questions. It uses its own scorer, which takes the last number in the reply, because a model might answer “42”, “17 + 25 = 42” or “The answer is 42.”:

The lesson installs this first:

In [ ]:
!pip install inspect-ai transformers accelerate

This cell needs a GPU runtime.

In [ ]:
%%writefile addition.py
# a tiny eval with its own scorer
import re
from inspect_ai import Task, task
from inspect_ai.dataset import MemoryDataset, Sample
from inspect_ai.scorer import (CORRECT, INCORRECT, Score, Target,
                               accuracy, scorer, stderr)
from inspect_ai.solver import TaskState, generate, system_message
@scorer(metrics=[accuracy(), stderr()])
def last_number():
    async def score(state: TaskState, target: Target) -> Score:
        found = re.findall(r"-?\d+", state.output.completion)
        answer = found[-1] if found else ""
        ok = answer == target.text
        return Score(value=CORRECT if ok else INCORRECT, answer=answer)
    return score
@task
def addition():
    pairs = [(17, 25), (230, 41), (9, 99)]
    samples = [Sample(input=f"What is {a} + {b}?", target=str(a + b))
               for a, b in pairs]
    return Task(dataset=MemoryDataset(samples),
                solver=[system_message("Reply with just the number."),
                        generate()],
                scorer=last_number())

### Running it

Use a Colab GPU runtime (Runtime > Change runtime type). Install Inspect in a cell with `!pip install inspect-ai transformers accelerate`, where `accelerate` is a Hugging Face helper that places a model on your GPU. Run the task on a small open model from the Hugging Face Hub with the `hf/` prefix, or with `vllm/` to generate through vLLM, after `!pip install vllm`:

This cell needs a GPU runtime.

In [ ]:
!inspect eval addition.py --model hf/Qwen/Qwen2.5-0.5B-Instruct
!inspect eval addition.py --model vllm/Qwen/Qwen2.5-0.5B-Instruct \
  --epochs 5 --temperature 0.8

Each run prints the accuracy and its standard error and writes a log file into a `logs` folder. Read every sample of it in the next cell:

This cell needs a GPU runtime.

In [ ]:
from inspect_ai.analysis import samples_df

df = samples_df("logs")      # one row per sample, from every log
print(df[["input", "target"]].head())

### Your own data, and ready-made benchmarks

For a real eval you will write a file with one question per line in JSON, called JSONL, and load it with `json_dataset`. If your fields have other names, `FieldSpec` maps them:

This cell needs a GPU runtime.

In [ ]:
from inspect_ai.dataset import FieldSpec, json_dataset

dataset = json_dataset("my_questions.jsonl",
                       FieldSpec(input="question", target="answer",
                                 id="qid"))

## 12.8 Project: an eval harness you own

The measuring tool every later module uses to prove a change helped.

[Read the lesson](https://aicourses.dev/research/inference-and-evals/project-eval-harness)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Steps

In [ ]:
import numpy as np
def report(a, b, seed=0):
    """a, b: per-question scores (0 to 1), lined up by sample id."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    n = len(a)
    for name, s in (("A", a), ("B", b)):
        print(f"{name}: {s.mean():.3f}, SE {s.std() / np.sqrt(n):.3f}")
    d = a - b
    se = d.std() / np.sqrt(n)
    lo, hi = d.mean() - 1.96 * se, d.mean() + 1.96 * se
    print(f"A - B: {d.mean():.3f}, paired 95% CI [{lo:.3f}, {hi:.3f}]")
    rng = np.random.default_rng(seed)
    boot = d[rng.integers(0, n, size=(10_000, n))].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"bootstrap 95% CI [{lo:.3f}, {hi:.3f}]")

# a self-check on made-up scores for 100 questions
rng = np.random.default_rng(1)
ease = 2 * rng.normal(size=100)
report(ease + rng.normal(size=100) > -1.0,
       ease + rng.normal(size=100) > -0.2)
# A: 0.710, SE 0.045
# B: 0.460, SE 0.050
# A - B: 0.250, paired 95% CI [0.149, 0.351]
# bootstrap 95% CI [0.150, 0.350]